# Baby Face Coverage Detection

## Overview
This notebook implements a two-stage pipeline for detecting whether a baby's face is covered or visible in video footage.

## Architecture
1. **Stage 1 - Face Detection**: Haar Cascade classifier locates the face region
2. **Stage 2 - Coverage Classification**: Texture analysis determines if the face is covered

## Key Metrics
- **Global variance**: Uniform textures (blankets) have low variance
- **Regional variance**: 3x3 grid analysis detects local uniformity
- **Edge density**: Facial features produce high edge density
- **Skin ratio**: HSV-based skin detection validates face visibility

## Decision Logic
A composite texture score below 0.35 indicates a covered face.
Temporal smoothing reduces flicker using a rolling history buffer.

In [ ]:
# Standard library imports
import os
from collections import deque

# Third-party imports
import cv2
import numpy as np

In [ ]:
class BabyFaceCoverageDetector:
    """
    Two-stage detector for baby face coverage.

    Stage 1: Haar Cascade face detection.
    Stage 2: Texture-based coverage classification.
    """

    # Classification thresholds
    COVERED_THRESHOLD = 0.35
    HISTORY_SIZE = 15
    SMOOTHING_MIN_SAMPLES = 5

    def __init__(self):
        """Initialize the face detector and classification parameters."""
        cascade_path = cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
        self.face_cascade = cv2.CascadeClassifier(cascade_path)

        if self.face_cascade.empty():
            raise RuntimeError('Failed to load Haar Cascade classifier.')

        # Temporal smoothing buffer
        self.coverage_history = deque(maxlen=self.HISTORY_SIZE)

        # Running statistics
        self.stats = {
            'total_frames': 0,
            'frames_with_face': 0,
            'frames_covered': 0,
        }

    def detect_face(self, frame):
        """
        Detect the largest valid face in the frame.

        Args:
            frame: BGR image array.

        Returns:
            Tuple (x, y, w, h) or None if no face is detected.
        """
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        gray = cv2.equalizeHist(gray)

        faces = self.face_cascade.detectMultiScale(
            gray,
            scaleFactor=1.1,
            minNeighbors=5,
            minSize=(80, 80),
            maxSize=(400, 400),
            flags=cv2.CASCADE_SCALE_IMAGE,
        )

        if len(faces) == 0:
            return None

        # Keep faces with a plausible aspect ratio (near square)
        valid_faces = [
            (x, y, w, h) for (x, y, w, h) in faces
            if 0.8 < (h / w) < 1.5
        ]

        if not valid_faces:
            return None

        # Return the largest face by area
        return max(valid_faces, key=lambda r: r[2] * r[3])

    def analyze_coverage(self, face_image):
        """
        Classify a face crop as covered or visible via texture analysis.

        Args:
            face_image: BGR image crop of the face region.

        Returns:
            Tuple (is_covered, confidence, details_dict).
        """
        if face_image is None or face_image.size == 0:
            return False, 0.0, {'score': 0.0, 'reason': 'empty_roi'}

        h, w = face_image.shape[:2]
        gray = cv2.cvtColor(face_image, cv2.COLOR_BGR2GRAY)

        # Metric 1: Global variance (rich textures indicate visible facial features)
        global_variance = float(np.var(gray))

        # Metric 2: Regional variance over a 3x3 grid
        region_variances = []
        for i in range(3):
            for j in range(3):
                y0, y1 = i * h // 3, (i + 1) * h // 3
                x0, x1 = j * w // 3, (j + 1) * w // 3
                region = gray[y0:y1, x0:x1]
                if region.size > 0:
                    region_variances.append(float(np.var(region)))
        avg_region_variance = float(np.mean(region_variances)) if region_variances else 0.0

        # Metric 3: Edge density via Canny
        edges = cv2.Canny(gray, 50, 150)
        edge_density = float(np.sum(edges > 0)) / (h * w)

        # Metric 4: Skin-color ratio in HSV space
        hsv = cv2.cvtColor(face_image, cv2.COLOR_BGR2HSV)
        lower_skin = np.array([0, 15, 50], dtype=np.uint8)
        upper_skin = np.array([25, 150, 255], dtype=np.uint8)
        skin_mask = cv2.inRange(hsv, lower_skin, upper_skin)
        skin_ratio = float(np.sum(skin_mask > 0)) / (h * w)

        # Normalize metrics to [0, 1]
        norm_global = min(global_variance / 5000.0, 1.0)
        norm_region = min(avg_region_variance / 3000.0, 1.0)
        norm_edges = min(edge_density * 20.0, 1.0)
        norm_skin = skin_ratio

        # Weighted composite score
        score = (
            norm_global * 0.30
            + norm_region * 0.30
            + norm_edges * 0.20
            + norm_skin * 0.20
        )

        # Classify with confidence tiers
        if score < 0.25:
            is_covered, confidence, reason = True, 0.90, 'uniform_texture'
        elif score < self.COVERED_THRESHOLD:
            is_covered, confidence, reason = True, 0.70, 'low_texture'
        elif score < 0.45:
            is_covered, confidence, reason = False, 0.60, 'uncertain'
        elif score < 0.60:
            is_covered, confidence, reason = False, 0.70, 'moderate_texture'
        else:
            is_covered, confidence, reason = False, 0.90, 'rich_texture'

        details = {
            'score': score,
            'global_var': global_variance,
            'region_var': avg_region_variance,
            'edge_density': edge_density,
            'skin_ratio': skin_ratio,
            'reason': reason,
        }

        return is_covered, confidence, details

    def process_frame(self, frame):
        """
        Run the full pipeline on a single frame and annotate it.

        Args:
            frame: BGR image array.

        Returns:
            Tuple (annotated_frame, is_covered, score).
        """
        self.stats['total_frames'] += 1

        face = self.detect_face(frame)

        if face is None:
            cv2.putText(
                frame, 'Searching for face...', (20, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2,
            )
            return frame, False, 0.0

        x, y, w, h = face
        self.stats['frames_with_face'] += 1

        face_image = frame[y:y + h, x:x + w]
        is_covered, confidence, details = self.analyze_coverage(face_image)

        # Temporal smoothing via majority vote
        self.coverage_history.append(is_covered)
        if len(self.coverage_history) >= self.SMOOTHING_MIN_SAMPLES:
            smoothed = sum(self.coverage_history) > len(self.coverage_history) / 2
        else:
            smoothed = is_covered

        if smoothed:
            self.stats['frames_covered'] += 1

        # Annotate the frame
        color = (0, 0, 255) if smoothed else (0, 255, 0)
        label = 'FACE COVERED' if smoothed else 'Face visible'

        cv2.rectangle(frame, (x, y), (x + w, y + h), color, 3)
        cv2.putText(frame, label, (x, y - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
        cv2.putText(frame, f'Confidence: {confidence:.0%}', (x, y + h + 15),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)
        cv2.putText(frame, f'Texture: {details["score"]:.2f}', (x, y + h + 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (200, 200, 200), 1)

        # Texture score bar
        bar_x, bar_y, bar_w, bar_h = x, y + h + 45, w, 8
        cv2.rectangle(frame, (bar_x, bar_y), (bar_x + bar_w, bar_y + bar_h), (50, 50, 50), -1)

        score = details['score']
        if score < 0.25:
            bar_color = (0, 0, 255)
        elif score < 0.35:
            bar_color = (0, 69, 255)
        elif score < 0.45:
            bar_color = (0, 165, 255)
        elif score < 0.60:
            bar_color = (0, 255, 255)
        else:
            bar_color = (0, 255, 0)

        bar_length = int(bar_w * min(score, 1.0))
        if bar_length > 0:
            cv2.rectangle(frame, (bar_x, bar_y), (bar_x + bar_length, bar_y + bar_h), bar_color, -1)

        return frame, smoothed, score

In [ ]:
def process_video(video_path, output_path=None):
    """
    Process a video file end-to-end and write an annotated output video.

    Args:
        video_path: Path to the input video.
        output_path: Optional output path; defaults to '<name>_annotated.mp4'.

    Returns:
        Output path on success, None on failure.
    """
    if not os.path.exists(video_path):
        print(f'Error: file not found: {video_path}')
        return None

    detector = BabyFaceCoverageDetector()

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print('Error: unable to open video.')
        return None

    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f'Video: {os.path.basename(video_path)}')
    print(f'  Resolution: {width}x{height}')
    print(f'  FPS: {fps}')
    print(f'  Duration: {total_frames / max(fps, 1):.1f}s')

    if output_path is None:
        name, ext = os.path.splitext(video_path)
        output_path = f'{name}_annotated{ext}'

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    print('Processing...')
    frame_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_count += 1
        processed, _, _ = detector.process_frame(frame)
        out.write(processed)

        if fps > 0 and frame_count % (fps * 2) == 0:
            progress = frame_count / max(total_frames, 1) * 100
            print(f'  Progress: {progress:.1f}% ({frame_count}/{total_frames})')

    cap.release()
    out.release()

    print(f'Output saved: {output_path}')

    stats = detector.stats
    if stats['frames_with_face'] > 0:
        covered_pct = stats['frames_covered'] / stats['frames_with_face'] * 100
        print('Statistics:')
        print(f'  Total frames:      {stats["total_frames"]}')
        print(f'  Frames with face:  {stats["frames_with_face"]}')
        print(f'  Frames covered:    {stats["frames_covered"]}')
        print(f'  Frames visible:    {stats["frames_with_face"] - stats["frames_covered"]}')
        print(f'  Covered ratio:     {covered_pct:.1f}%')

    return output_path

In [ ]:
# Entry point: set the input path and run the pipeline.
VIDEO_PATH = r'C:\Users\mouni\Documents\ALMA_project\babyFace_convered_Uncovered\video7.mp4'

output = process_video(VIDEO_PATH)
print(f'\nResult: {output}')